In [8]:
from pathlib import Path
import pandas as pd
import soundfile as sf
from IPython.display import display

In [9]:
# Đường dẫn gốc tới dataset VNEMOS
DATA_DIR = Path("data/VNEMOS")

# Các đuôi file audio hỗ trợ
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

print("DATA_DIR =", DATA_DIR.resolve())
print("Exists   =", DATA_DIR.exists())

DATA_DIR = /home/emotalk/mer2/data/VNEMOS
Exists   = True


In [10]:
def get_audio_duration_seconds(audio_path: Path) -> float:
    """
    Trả về độ dài file audio theo giây mà không cần load toàn bộ waveform vào RAM.
    """
    info = sf.info(str(audio_path))
    return float(info.frames) / float(info.samplerate)

In [11]:
records = []

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu: {DATA_DIR}")

for label_dir in sorted(DATA_DIR.iterdir()):
    if not label_dir.is_dir():
        continue

    label = label_dir.name

    for audio_path in sorted(label_dir.rglob("*")):
        if audio_path.is_file() and audio_path.suffix.lower() in AUDIO_EXTENSIONS:
            try:
                duration_sec = get_audio_duration_seconds(audio_path)

                records.append({
                    "label": label,
                    "file_name": audio_path.name,
                    "file_path": str(audio_path),
                    "duration_sec": duration_sec,
                })
            except Exception as e:
                print(f"[WARNING] Không đọc được file: {audio_path} | Error: {e}")

df = pd.DataFrame(records)

print(f"Tổng số file hợp lệ: {len(df)}")
display(df.head())

Tổng số file hợp lệ: 250


,label,file_name,file_path,duration_sec
0,angry,Copy of Angry_scvmc12-00.16.21.029-00.16.25.28...,data/VNEMOS/angry/Copy of Angry_scvmc12-00.16....,4.783311
1,angry,Copy of Angry_scvmc15-00.09.41.333-00.09.43.73...,data/VNEMOS/angry/Copy of Angry_scvmc15-00.09....,2.461315
2,angry,Copy of Angry_scvmc15-00.09.45.505-00.09.52.32...,data/VNEMOS/angry/Copy of Angry_scvmc15-00.09....,10.402540
3,angry,Copy of Angry_scvmc15-00.33.11.861-00.33.18.08...,data/VNEMOS/angry/Copy of Angry_scvmc15-00.33....,8.823583
4,angry,Copy of Angry_scvmc16-00.02.12.124-00.02.17.51...,data/VNEMOS/angry/Copy of Angry_scvmc16-00.02....,6.176508


In [12]:
if df.empty:
    print("Không tìm thấy file audio hợp lệ nào.")
else:
    label_counts = (
        df["label"]
        .value_counts()
        .rename_axis("label")
        .reset_index(name="num_samples")
        .sort_values("label")
        .reset_index(drop=True)
    )

    print("Số mẫu theo label:")
    display(label_counts)

Số mẫu theo label:


,label,num_samples
0,angry,50
1,fear,50
2,happiness,50
3,neutral,50
4,sadness,50


In [13]:
if df.empty:
    print("Không có dữ liệu để thống kê.")
else:
    summary_df = (
        df.groupby("label")
          .agg(
              num_samples=("duration_sec", "count"),
              avg_duration_sec=("duration_sec", "mean"),
              min_duration_sec=("duration_sec", "min"),
              max_duration_sec=("duration_sec", "max"),
          )
          .reset_index()
          .sort_values("label")
          .reset_index(drop=True)
    )

    # Làm tròn để dễ đọc
    summary_df["avg_duration_sec"] = summary_df["avg_duration_sec"].round(3)
    summary_df["min_duration_sec"] = summary_df["min_duration_sec"].round(3)
    summary_df["max_duration_sec"] = summary_df["max_duration_sec"].round(3)

    print("Thống kê theo label:")
    display(summary_df)

Thống kê theo label:


,label,num_samples,avg_duration_sec,min_duration_sec,max_duration_sec
0,angry,50,5.476,1.152,12.423
1,fear,50,5.751,1.277,18.831
2,happiness,50,9.291,4.203,17.600
3,neutral,50,8.626,3.042,20.875
4,sadness,50,10.794,4.365,30.883


In [14]:
if df.empty:
    print("Không có dữ liệu để thống kê.")
else:
    overall_stats = {
        "total_samples": len(df),
        "num_labels": df["label"].nunique(),
        "avg_duration_sec": round(df["duration_sec"].mean(), 3),
        "min_duration_sec": round(df["duration_sec"].min(), 3),
        "max_duration_sec": round(df["duration_sec"].max(), 3),
    }

    overall_stats_df = pd.DataFrame([overall_stats])
    print("Thống kê tổng quan toàn bộ VNEMOS:")
    display(overall_stats_df)

Thống kê tổng quan toàn bộ VNEMOS:


,total_samples,num_labels,avg_duration_sec,min_duration_sec,max_duration_sec
0,250,5,7.988,1.152,30.883
